# Grover's search
**Quantum Computing · Week 5 (Quantum protocols and algorithms) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w5d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Project: Grover's search.** One item out of $N = 2^n$ is marked. Grover's algorithm finds it with
about $\frac{\pi}{4}\sqrt{N}$ oracle calls instead of about $N/2$. You'll build the oracle, the
diffuser and the full search, then find out how many steps to take.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler

def phase_oracle(marked):
    """Flip the sign of one basis state. marked is a bit string in Qiskit order (last character = qubit 0)."""
    n = len(marked)
    qc = QuantumCircuit(n, name="oracle")
    zeros = [q for q, bit in enumerate(reversed(marked)) if bit == "0"]
    if zeros:
        qc.x(zeros)                          # now the marked state looks like 11...1
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)        # H, multi-controlled X, H = multi-controlled Z
    qc.h(n - 1)
    if zeros:
        qc.x(zeros)
    return qc

def diffuser(n):
    """Reflect every amplitude about the mean (up to an overall sign, which can't be measured)."""
    qc = QuantumCircuit(n, name="diffuser")
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

def grover(marked, steps, measure=True):
    n = len(marked)
    qc = QuantumCircuit(n)
    qc.h(range(n))                           # every item equally likely
    oracle_gate, diffuser_gate = phase_oracle(marked).to_gate(), diffuser(n).to_gate()
    for _ in range(steps):
        qc.append(oracle_gate, range(n))     # appended as boxes, so the drawing stays readable
        qc.append(diffuser_gate, range(n))
    if measure:
        qc.measure_all()
    return qc

display(grover("101", 2).draw("mpl", style="clifford"))
diffuser(3).draw("mpl", style="clifford")

## Search 8 items

In [ ]:
sampler = StatevectorSampler(seed=1)
for steps in [0, 1, 2]:
    counts = sampler.run([grover("101", steps)], shots=1000).result()[0].data.meas.get_counts()
    print(f"{steps} Grover steps: '101' found {counts.get('101', 0)} times out of 1000")

## More is not better
The success probability follows $\sin^2\big((2k + 1)\theta\big)$ with $\sin\theta = 1/\sqrt{N}$:
each step turns the state by $2\theta$ towards the marked item, and too many steps turn it past.

In [ ]:
N = 8
theta = np.arcsin(1 / np.sqrt(N))
ks = np.arange(0, 11)
simulated = [Statevector(grover("101", k, measure=False)).probabilities_dict()["101"] for k in ks]
fine = np.linspace(0, 10, 300)

fig, ax = plt.subplots(figsize=(6.4, 3.2), layout="constrained")
ax.plot(fine, np.sin((2 * fine + 1) * theta) ** 2, color="#b9a7ff", lw=2, label=r"$\sin^2((2k+1)\theta)$")
ax.plot(ks, simulated, "o", color="#7b5cfa", ms=7, label="simulated circuit")
ax.set_xlabel("Grover steps k")
ax.set_ylabel("P(marked item)")
ax.set_title("N = 8: best after 2 steps, then it comes back down")
ax.legend(loc="lower right")
plt.show()

## Project: how many steps for N items?
1. For $n = 2, \dots, 8$ qubits, simulate the success probability for $k = 0, 1, 2, \dots$ steps
   (use `Statevector(grover(marked, k, measure=False))`) and find the best $k$.
2. Compare it with $\frac{\pi}{4}\sqrt{N}$, and with $N/2$, the average number of boxes a classical
   search opens.
3. Plot the best $k$ against $\sqrt{N}$. What shape do you expect?

In [ ]:
rows = []
for n in range(2, 9):
    N = 2 ** n
    marked = "1" * n                         # any marked string works the same way
    best_k, best_p = 0, 0.0
    # TODO: try k = 0, 1, ... up to about pi/4 * sqrt(N) + 2, keep the k with the highest probability
    rows.append((n, N, best_k, best_p))
for n, N, best_k, best_p in rows:
    print(f"n = {n}, N = {N:3d}: best k = {best_k} (P = {best_p:.3f}), pi/4 sqrt(N) = {np.pi / 4 * np.sqrt(N):.2f}, classical N/2 = {N // 2}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
rows = []
for n in range(2, 9):
    N = 2 ** n
    marked = "1" * n
    probs = [Statevector(grover(marked, k, measure=False)).probabilities_dict().get(marked, 0.0)
             for k in range(int(np.pi / 4 * np.sqrt(N)) + 3)]
    best_k = int(np.argmax(probs))
    rows.append((n, N, best_k, probs[best_k]))
for n, N, best_k, best_p in rows:
    print(f"n = {n}, N = {N:3d}: best k = {best_k:2d} (P = {best_p:.3f}), pi/4 sqrt(N) = {np.pi / 4 * np.sqrt(N):5.2f}, classical N/2 = {N // 2}")

sqrtN = [np.sqrt(N) for _, N, _, _ in rows]
fig, ax = plt.subplots(figsize=(5.6, 3.2), layout="constrained")
ax.plot(sqrtN, [k for _, _, k, _ in rows], "o", color="#7b5cfa", ms=8, label="best k (simulated)")
ax.plot(sqrtN, np.pi / 4 * np.array(sqrtN), color="#b9a7ff", lw=2, label=r"$\frac{\pi}{4}\sqrt{N}$")
ax.set_xlabel(r"$\sqrt{N}$")
ax.set_ylabel("Grover steps")
ax.legend()
plt.show()
print("A straight line in sqrt(N): searching 100 times more items takes only 10 times more steps.")